# WhisperBench — Audio Dataset Quality Pipeline + LoRA Fine-Tuning

**Part 1 (Cells 1–8):** Ingest raw audio, transcribe via Whisper, score quality metrics, filter, export Parquet.

**Part 2 (Cells 9–16):** Baseline WER → LoRA fine-tune Whisper-small on 2,000 samples → WER after.




In [1]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
!pip install -q openai-whisper pandas pyarrow tqdm soundfile scipy \
                transformers datasets peft accelerate jiwer evaluate
!apt-get install -q ffmpeg
import whisper, numpy, pandas, scipy, transformers, peft
print(f"numpy {numpy.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 48.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
numpy 2.0.2 | transformers 5.0.0 | peft 0.18.1


In [2]:
# ── Cell 2: Download LibriSpeech test-clean ───────────────────────────────────
import os, tarfile, urllib.request

URL  = "https://www.openslr.org/resources/12/test-clean.tar.gz"
DEST = "test-clean.tar.gz"

if not os.path.exists("LibriSpeech"):
    print("Downloading LibriSpeech test-clean (~346 MB)...")
    urllib.request.urlretrieve(URL, DEST)
    print("Extracting...")
    with tarfile.open(DEST) as t:
        t.extractall()
    os.remove(DEST)
    print("Done.")
else:
    print("Already downloaded.")

Extracting...


/tmp/ipykernel_3247/3467674184.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall()


Done.


In [3]:
# ── Cell 3: Collect files ─────────────────────────────────────────────────────
import glob

audio_files = sorted(glob.glob("LibriSpeech/**/*.flac", recursive=True))
print(f"Found {len(audio_files)} audio files")

Found 2620 audio files


In [4]:
# ── Cell 4: Load Whisper base (quality pipeline) ──────────────────────────────
import whisper
pipe_model = whisper.load_model("base")
print("Whisper base loaded")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 239MiB/s]


Whisper base loaded


In [5]:
# ── Cell 5: Scoring functions ─────────────────────────────────────────────────
import numpy as np
import soundfile as sf

def load_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    return y, sr

def compute_snr(y):
    frame_size = 1024
    frames = [y[i:i+frame_size] for i in range(0, len(y)-frame_size, frame_size)]
    if not frames:
        return 0.0
    rms = np.array([np.sqrt(np.mean(f**2)) for f in frames])
    noise = np.percentile(rms, 10)
    signal = np.mean(rms)
    if noise < 1e-10:
        return 99.0
    return round(float(20 * np.log10(signal / noise)), 2)

def get_duration(y, sr):
    return round(float(len(y) / sr), 3)

def transcribe_pipe(path):
    result = pipe_model.transcribe(path, language="en", fp16=True)
    text = result["text"].strip()
    segs = result.get("segments", [])
    avg_logprob = float(np.mean([float(s["avg_logprob"]) for s in segs])) if segs else -1.0
    return text, round(avg_logprob, 4)

print("Scoring functions ready")

Scoring functions ready


In [6]:
# ── Cell 6: Run quality pipeline ──────────────────────────────────────────────
# LIMIT = 100 for quick test, None for full run (~23 min)
import time
from tqdm import tqdm

LIMIT = 100

files_to_process = audio_files[:LIMIT] if LIMIT else audio_files

records, errors = [], []
t0 = time.time()

for path in tqdm(files_to_process, desc="Processing"):
    try:
        y, sr      = load_audio(path)
        duration   = get_duration(y, sr)
        snr        = compute_snr(y)
        transcript, confidence = transcribe_pipe(path)
        records.append({
            "file":       os.path.basename(path),
            "duration_s": duration,
            "snr_db":     snr,
            "transcript": transcript,
            "confidence": float(confidence),
        })
    except Exception as e:
        errors.append({"file": path, "error": str(e)})

elapsed = time.time() - t0
print(f"\nProcessed {len(records)} files in {elapsed/60:.1f} min")
print(f"Errors:     {len(errors)}")
print(f"Throughput: {len(records)/(elapsed/60):.1f} files/min")
if errors:
    print(f"First error: {errors[0]}")

Processing: 100%|██████████| 100/100 [00:47<00:00,  2.09it/s]


Processed 100 files in 0.8 min
Errors:     0
Throughput: 125.2 files/min


In [7]:
# ── Cell 7: Filter & export Parquet ──────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(records)

MIN_DURATION_S = 1.0
MAX_DURATION_S = 30.0
MIN_SNR_DB     = 10.0
MIN_CONFIDENCE = -0.8
MIN_WORDS      = 2

df["word_count"] = df["transcript"].apply(lambda t: len(t.split()))

mask = (
    df["duration_s"].between(MIN_DURATION_S, MAX_DURATION_S) &
    (df["snr_db"]     >= MIN_SNR_DB) &
    (df["confidence"] >= MIN_CONFIDENCE) &
    (df["word_count"] >= MIN_WORDS)
)

df_clean    = df[mask].drop(columns=["word_count"])
df_rejected = df[~mask].drop(columns=["word_count"])

total_hours = df["duration_s"].sum() / 3600
clean_hours = df_clean["duration_s"].sum() / 3600
filter_rate = len(df_rejected) / len(df) * 100

print(f"Total files:   {len(df)}")
print(f"Total audio:   {total_hours:.2f} hrs")
print(f"Passed filter: {len(df_clean)} files ({clean_hours:.2f} hrs)")
print(f"Rejected:      {len(df_rejected)} files ({filter_rate:.1f}%)")

df_clean.to_parquet("whisperbench_clean.parquet", index=False)
df_rejected.to_parquet("whisperbench_rejected.parquet", index=False)
print("Exported parquet files.")

Total files:   100
Total audio:   0.25 hrs
Passed filter: 97 files (0.25 hrs)
Rejected:      3 files (3.0%)
Exported parquet files.


In [8]:
# ── Cell 8: Quality summary (screenshot this) ─────────────────────────────────
print("=== WHISPERBENCH QUALITY SUMMARY ===")
print(f"Files processed:  {len(df)}")
print(f"Total audio:      {total_hours:.2f} hrs")
print(f"Clean dataset:    {len(df_clean)} files / {clean_hours:.2f} hrs")
print(f"Filter rate:      {filter_rate:.1f}%")
print(f"Avg SNR (clean):  {df_clean['snr_db'].mean():.1f} dB")
print(f"Avg confidence:   {df_clean['confidence'].mean():.3f}")
print(f"Throughput:       {len(records)/(elapsed/60):.1f} files/min")
print()
print(df_clean[["file","duration_s","snr_db","confidence","transcript"]].head(5).to_string(index=False))

=== WHISPERBENCH QUALITY SUMMARY ===
Files processed:  100
Total audio:      0.25 hrs
Clean dataset:    97 files / 0.25 hrs
Filter rate:      3.0%
Avg SNR (clean):  26.2 dB
Avg confidence:   -0.308
Throughput:       125.2 files/min

                 file  duration_s  snr_db  confidence                                                                                                                                                      transcript
1089-134686-0000.flac      10.435   23.66     -0.2201 He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered flour-fatten sauce.
1089-134686-0001.flac       3.275   22.98     -0.6146                                                                                                                       Stuffed into you, his belly countled him.
1089-134686-0002.flac       6.625   31.37     -0.2268                                                     After early nightfa

---
# Part 2: LoRA Fine-Tuning
Fine-tune Whisper-small on 2,000 samples from LibriSpeech using LoRA/PEFT. Measure WER before vs after.

In [9]:
# ── Cell 9: Load Whisper-small + eval data + WER metric ──────────────────────
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
import evaluate
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

MODEL_ID = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
base_model.config.forced_decoder_ids = None

print("Loading eval split (100 samples from test-clean)...")
eval_ds = load_dataset("librispeech_asr", "clean", split="test[:100]")
print(f"Eval samples: {len(eval_ds)}")

wer_metric = evaluate.load("wer")
print("Ready.")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]

Eval samples: 100


Ready.


In [10]:
# ── Cell 10: Define WER functions ─────────────────────────────────────────────
def compute_wer_base(model, dataset, label="Model"):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for sample in tqdm(dataset, desc=f"Evaluating {label}"):
            audio  = sample["audio"]["array"]
            sr     = sample["audio"]["sampling_rate"]
            ref    = sample["text"].lower().strip()
            inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.to(DEVICE)
            predicted_ids = model.generate(inputs)
            hyp = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].lower().strip()
            refs.append(ref)
            hyps.append(hyp)
    wer = wer_metric.compute(predictions=hyps, references=refs)
    print(f"\n{label} WER: {wer*100:.2f}%")
    return wer

def compute_wer_peft(model, dataset, label="Model"):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for sample in tqdm(dataset, desc=f"Evaluating {label}"):
            audio  = sample["audio"]["array"]
            sr     = sample["audio"]["sampling_rate"]
            ref    = sample["text"].lower().strip()
            inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.to(DEVICE)
            predicted_ids = model.generate(input_features=inputs)  # keyword arg for PEFT
            hyp = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].lower().strip()
            refs.append(ref)
            hyps.append(hyp)
    wer = wer_metric.compute(predictions=hyps, references=refs)
    print(f"\n{label} WER: {wer*100:.2f}%")
    return wer

print("WER functions ready")

WER functions ready


In [11]:
# ── Cell 11: Baseline WER ─────────────────────────────────────────────────────
baseline_wer = compute_wer_base(base_model, eval_ds, label="Baseline")

Evaluating Baseline:   0%|          | 0/100 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para


Baseline WER: 16.84%


In [12]:
# ── Cell 12: Load & preprocess 2000 training samples ─────────────────────────
print("Loading 2000 training samples...")
train_ds = load_dataset("librispeech_asr", "clean", split="train.100[:2000]")
print(f"Loaded: {len(train_ds)} samples")

def preprocess(batch):
    audio = batch["audio"]
    inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = train_ds.map(preprocess, remove_columns=train_ds.column_names)
print(f"Done. {len(train_ds)} samples ready.")

Loading 2000 training samples...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loaded: 2000 samples


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Done. 2000 samples ready.


In [13]:
# ── Cell 13: Apply LoRA ───────────────────────────────────────────────────────
from peft import get_peft_model, LoraConfig, TaskType

ft_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
ft_model.config.forced_decoder_ids = None

ft_model = get_peft_model(ft_model, LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
))
ft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 442,368 || all params: 242,177,280 || trainable%: 0.1827


In [14]:
# ── Cell 14: Data collator ────────────────────────────────────────────────────
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Collator ready")

Collator ready


In [15]:
# ── Cell 15: Train (~20-25 min on A100) ──────────────────────────────────────
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-out",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    num_train_epochs=2,
    fp16=True,
    eval_strategy="no",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
    processing_class=processor.feature_extractor,
)

print("Training...")
trainer.train()
print("Done.")

Training...


Step,Training Loss
50,1.277433
100,0.859568
150,0.653458
200,0.567496
250,0.549326


Done.


In [19]:
# ── Cell 16: WER after + summary (screenshot this) ────────────────────────────
ft_wer = compute_wer_peft(ft_model, eval_ds, label="LoRA Fine-tuned")
improvement = (baseline_wer - ft_wer) / baseline_wer * 100
trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in ft_model.parameters())

print()
print("=== FINE-TUNING SUMMARY ===")
print(f"Baseline WER:      {baseline_wer*100:.2f}%")
print(f"Fine-tuned WER:    {ft_wer*100:.2f}%")
print(f"WER improvement:   {improvement:.1f}%")
print(f"LoRA rank:         4  |  target: q_proj, v_proj")
print(f"Trainable params:  {trainable/1e6:.1f}M / {total/1e6:.0f}M ({trainable/total*100:.1f}%)")
print(f"Training samples:  2,000")

Evaluating LoRA Fine-tuned: 100%|██████████| 100/100 [03:00<00:00,  1.80s/it]


LoRA Fine-tuned WER: 36.20%

=== FINE-TUNING SUMMARY ===
Baseline WER:      16.84%
Fine-tuned WER:    36.20%
WER improvement:   -114.9%
LoRA rank:         4  |  target: q_proj, v_proj
Trainable params:  0.4M / 242M (0.2%)
Training samples:  2,000


In [ ]:
# ── Cell 17: Save adapter weights ────────────────────────────────────────────
ft_model.save_pretrained("./whisper-lora-adapter")
processor.save_pretrained("./whisper-lora-adapter")


In [25]:
# ── Cell 18: Zip everything for download ─────────────────────────────────────
from google.colab import files
import shutil, os

# Zip all outputs together
shutil.make_archive("whisperbench_all", "zip", "/content", ".")
files.download("/content/whisperbench_all.zip")

RuntimeError: File size too large, try using force_zip64

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
import shutil, os

dest = "/content/drive/MyDrive/WhisperBench"
os.makedirs(dest, exist_ok=True)

# Copy key files
shutil.copy("whisperbench_clean.parquet", dest)
shutil.copy("whisperbench_rejected.parquet", dest)
shutil.copytree("whisper-lora-adapter", f"{dest}/whisper-lora-adapter", dirs_exist_ok=True)
shutil.copytree("whisper-lora-out", f"{dest}/whisper-lora-out", dirs_exist_ok=True)

print("Done! Check Google Drive → WhisperBench folder.")

Done! Check Google Drive → WhisperBench folder.
